# Qwen-27B Cross-Context Factor Equivalence and Product Interaction

Cross three source contexts with three target contexts to test whether factor updates follow the target's complementary factor.

[实验设置与运行说明](../../../docs/experiment_notes/10_exploratory_composition_and_gating/4a01bdcd271e7b01.md)


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 17 tracked cells.')


## 1. Setup


In [ ]:
%%cell_progress 1/17 Mount Drive and locate VisualTorque
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


In [ ]:
%%cell_progress 2/17 Install and verify the multimodal environment
import importlib
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)
cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34', 'tqdm>=4.66',
    'pandas>=2.2', 'matplotlib>=3.8', 'scikit-learn>=1.5',
])

cell_stage('Registering the live VisualTorque source tree')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
assert (PACKAGE_ROOT / '__init__.py').is_file(), PACKAGE_ROOT
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if (
        module_name == 'visual_torque_toolkit'
        or module_name.startswith('visual_torque_toolkit.')
    ):
        del sys.modules[module_name]
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
import visual_torque_toolkit
from IPython.display import display
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter

required_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(
    hasattr(HuggingFaceMultimodalAdapter, name)
    for name in required_features
)
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents, toolkit_path
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('Toolkit source:', toolkit_path)


In [ ]:
%%cell_progress 3/17 Verify GPU and Hugging Face access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Select an A100 GPU runtime.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(
    torch.cuda.get_device_properties(0).total_memory / 2**30, 1
))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Public checkpoints may still work.')
except Exception as exc:
    print('Hugging Face login skipped:', exc)


In [ ]:
%%cell_progress 4/17 Lock the factor-equivalence protocol
cell_stage('Freezing model, sites, support, estimands, and decision rules')
import hashlib
import json
from datetime import datetime, timezone

MODEL_ID = 'Qwen/Qwen3.5-27B'
MODEL_SLUG = 'Qwen__Qwen3.5-27B__373612cb'
MODEL_REVISION = '373612cb4f9288fcf550b526d86d85febedd9801'
MODEL_DTYPE = 'bfloat16'
DEVICE_MAP = 'auto'
EXPECTED_LAYER_COUNT = 64
UPSTREAM_LAYER = 48
UPSTREAM_SCOPE = 'options_tokens'
CANDIDATE_LAYERS = (49, 57)

SEED = 20261021
FACTORS = ('weight', 'distance')
SIDES = ('left', 'right')
SOURCE_KEYS = ('s1', 's2', 's3')
TARGET_KEYS = ('t1', 't2', 't3')
CANDIDATE_FAMILIES_PER_FACTOR = 64
CONFIRMATION_FAMILIES_PER_FACTOR = 16
EXPECTED_FAMILIES = len(FACTORS) * CONFIRMATION_FAMILIES_PER_FACTOR
MAPPING_VARIANTS = ('forward', 'reverse')
SITE_BATCH_SIZE = 4
BOOTSTRAP_REPLICATES = 10000
ALPHA = 0.05
SOURCE_EQUIVALENCE_RATIO = 0.25
ANSWER_EQUIVALENCE_RATIO = 0.25
MIN_TARGET_COUNTERFACTUAL_NORM = 0.05
SMOKE_CENTERED_LOGPROB_TOLERANCE = 0.15
NORM_RELATIVE_TOLERANCE = 1e-3
IMAGE_TOKEN_ID_OVERRIDE = None

RAW_CONDITIONS = tuple(f'raw_source_{key}' for key in SOURCE_KEYS)
RESIDUAL_CONDITIONS = tuple(f'residual_{key}' for key in SOURCE_KEYS)
WHOLE_CONDITIONS = tuple(f'whole_source_{key}' for key in SOURCE_KEYS)
CONDITIONS = (
    'clean', 'natural_hybrid', *RAW_CONDITIONS, 'shared',
    *RESIDUAL_CONDITIONS, 'reverse_shared', 'random_shared',
    *WHOLE_CONDITIONS, 'self_replay', 'joint_restore', 'joint_random',
)

PRIOR_ROUTE_ROOT = (
    PROJECT_ROOT / 'results' / 'qwen27b_factor_path_token_component_v2'
)
PRIOR_RUN_PATH = PRIOR_ROUTE_ROOT / 'RUN_COMPLETE.json'
PRIOR_SELECTION_PATH = PRIOR_ROUTE_ROOT / 'analysis' / 'frozen_sites.json'
PRIOR_GRADED_MANIFEST_PATH = (
    PROJECT_ROOT / 'results' / 'qwen27b_graded_product_law_v1'
    / 'graded_product_manifest.jsonl'
)

SHARED_ROOT = (
    PROJECT_ROOT / 'results'
    / 'qwen27b_crosscontext_factor_equivalence_product_v1'
)
CANDIDATE_MANIFEST_PATH = SHARED_ROOT / 'candidate_manifest.jsonl'
CANDIDATE_LOCK_PATH = SHARED_ROOT / 'candidate_manifest_lock.json'
SCREEN_DIR = SHARED_ROOT / 'endpoint_screen'
CONFIRMATION_MANIFEST_PATH = SHARED_ROOT / 'confirmation_manifest.jsonl'
CONFIRMATION_LOCK_PATH = SHARED_ROOT / 'confirmation_manifest_lock.json'
PROTOCOL_LOCK_PATH = SHARED_ROOT / 'frozen_protocol.json'
SHARD_DIR = SHARED_ROOT / 'family_shards'
ANALYSIS_DIR = SHARED_ROOT / 'analysis'
FIGURE_DIR = ANALYSIS_DIR / 'figures'
for path in (
    SHARED_ROOT, SCREEN_DIR, SHARD_DIR, ANALYSIS_DIR, FIGURE_DIR,
):
    path.mkdir(parents=True, exist_ok=True)

def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f'Object of type {type(value).__name__} is not JSON serializable')

def write_json_atomic(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(
            value, indent=2, ensure_ascii=False, default=json_default
        ) + '\n',
        encoding='utf-8',
    )
    temporary.replace(path)

def write_jsonl_atomic(path, rows):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(
                row, sort_keys=True, default=json_default
            ) + '\n')
    temporary.replace(path)

def read_jsonl(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

def stable_key(value):
    return hashlib.sha256(f'{SEED}:{value!r}'.encode('utf-8')).hexdigest()

for required in (PRIOR_RUN_PATH, PRIOR_SELECTION_PATH):
    assert required.is_file(), f'Missing prerequisite: {required}'
prior_run = json.loads(PRIOR_RUN_PATH.read_text(encoding='utf-8'))
prior_selection = json.loads(
    PRIOR_SELECTION_PATH.read_text(encoding='utf-8')
)
assert prior_run['primary_decision'] == (
    'sparse_token_component_route_confirmed_for_both_factors'
)
assert prior_selection['confirmation_used_for_selection'] is False
FROZEN_SITES = prior_selection['sites']
expected_signature = (
    (49, 'decoder', 'options_tokens'),
    (57, 'decoder', 'final_prompt_token'),
)
for factor in FACTORS:
    observed = tuple(
        (int(site['layer_index']), str(site['component']), str(site['scope']))
        for site in FROZEN_SITES[factor]
    )
    assert observed == expected_signature, (factor, observed)

protocol_identity = {
    'schema_version': 'factor-equivalence-product-1.0',
    'analysis_status': 'frozen competence-matched confirmation',
    'model_id': MODEL_ID,
    'revision': MODEL_REVISION,
    'upstream_site': [UPSTREAM_LAYER, UPSTREAM_SCOPE],
    'frozen_sites': FROZEN_SITES,
    'factors': list(FACTORS),
    'source_contexts_per_family': len(SOURCE_KEYS),
    'target_contexts_per_family': len(TARGET_KEYS),
    'candidate_families_per_factor': CANDIDATE_FAMILIES_PER_FACTOR,
    'confirmation_families_per_factor': CONFIRMATION_FAMILIES_PER_FACTOR,
    'conditions': list(CONDITIONS),
    'primary_crossed_model': (
        'target-counterfactual progress ~ target identity + source physical '
        'dose + source-answer agreement, clustered by family'
    ),
    'factor_equivalence_gate': (
        'positive progress toward target-specific natural composites; upper '
        'confidence bounds for source-context and source-answer nuisance '
        'effects relative to that progress below '
        f'{SOURCE_EQUIVALENCE_RATIO}'
    ),
    'shared_component_gate': (
        'shared target-dose slope positive and larger than source-specific '
        'residual and equal-norm random slopes'
    ),
    'route_gate': (
        'frozen L49/L57 restoration attenuates progress toward each natural '
        'target composite '
        'more than equal-norm random restoration'
    ),
    'selection_policy': (
        'model-blind physical candidates followed by all-endpoint-correct '
        'batch-one competence screen; intervention outcomes never inspected'
    ),
    'source_equivalence_ratio': SOURCE_EQUIVALENCE_RATIO,
    'answer_equivalence_ratio': ANSWER_EQUIVALENCE_RATIO,
    'minimum_target_counterfactual_logprob_norm': MIN_TARGET_COUNTERFACTUAL_NORM,
    'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    'prior_route_selection_sha256': sha256_file(PRIOR_SELECTION_PATH),
    'seed': SEED,
}
if PROTOCOL_LOCK_PATH.is_file():
    previous = json.loads(PROTOCOL_LOCK_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == protocol_identity, (
        'Existing protocol differs. Use a new versioned output directory.'
    )
else:
    write_json_atomic(PROTOCOL_LOCK_PATH, {
        'identity': protocol_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })

print('Model:', MODEL_ID, '@', MODEL_REVISION)
print('Frozen route:', expected_signature)
print('Candidate / confirmation families per factor:',
      CANDIDATE_FAMILIES_PER_FACTOR, CONFIRMATION_FAMILIES_PER_FACTOR)
print('Output:', SHARED_ROOT)


In [ ]:
%%cell_progress 5/17 Prepare Dataset v1 on local SSD
cell_stage('Checking local and Drive dataset caches')
from zipfile import ZipFile

REQUIRED_DATASET_FILES = {
    'dataset_config.json', 'scenes.jsonl', 'counterfactual_pairs.jsonl',
    'questions.jsonl', 'splits.json', 'statistics.json',
}
DATASET_DIR = Path('/content/visualtorque_dataset_v1')
DATASET_ARCHIVE = PROJECT_ROOT / 'data' / 'dataset_v1.zip'
LEGACY_DATASET_DIR = PROJECT_ROOT / 'data' / 'dataset_v1'

def dataset_complete(path):
    return path.is_dir() and all(
        (path / name).is_file() for name in REQUIRED_DATASET_FILES
    )

if not dataset_complete(DATASET_DIR):
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if DATASET_ARCHIVE.is_file():
        cell_stage('Extracting Dataset v1 archive')
        with ZipFile(DATASET_ARCHIVE, 'r') as bundle:
            members = [m for m in bundle.infolist() if not m.is_dir()]
            root = DATASET_DIR.resolve()
            for member in tqdm(members, desc='Extracting Dataset v1', unit='file'):
                target = (DATASET_DIR / member.filename).resolve()
                if root not in target.parents:
                    raise ValueError(f'Unsafe archive path: {member.filename}')
                bundle.extract(member, DATASET_DIR)
    elif dataset_complete(LEGACY_DATASET_DIR):
        DATASET_DIR = LEGACY_DATASET_DIR
    else:
        raise FileNotFoundError(
            'Run the Dataset v1 build/cache cell in the behavior notebook first.'
        )
assert dataset_complete(DATASET_DIR)
print('Dataset:', DATASET_DIR)


In [ ]:
%%cell_progress 6/17 Build crossed model-blind candidate families
cell_stage('Enumerating balanced three-source by three-target families')
from collections import Counter, defaultdict

from visual_torque_toolkit.behavior import (
    BehaviorRunConfig, PromptCondition, load_behavior_examples,
)

scene_records = read_jsonl(DATASET_DIR / 'scenes.jsonl')
scene_by_id = {str(row['scene_id']): row for row in scene_records}

def single_shape(scene):
    if len(scene['stacks']) != 2:
        return None
    left = [stack for stack in scene['stacks'] if stack['side'] == 'left']
    right = [stack for stack in scene['stacks'] if stack['side'] == 'right']
    if len(left) != 1 or len(right) != 1:
        return None
    return (
        int(left[0]['units']), int(left[0]['distance']),
        int(right[0]['units']), int(right[0]['distance']),
    )

scene_by_shape = {}
for scene in scene_records:
    shape = single_shape(scene)
    if shape is not None:
        assert shape not in scene_by_shape
        scene_by_shape[shape] = str(scene['scene_id'])
assert len(scene_by_shape) == 900, len(scene_by_shape)

prior_scene_ids = set()
if PRIOR_GRADED_MANIFEST_PATH.is_file():
    for row in read_jsonl(PRIOR_GRADED_MANIFEST_PATH):
        prior_scene_ids.update(
            str(role['scene_id']) for role in row['roles'].values()
        )

def opposite_side(side):
    return 'right' if side == 'left' else 'left'

def shape_for(side, units, distance, opposite_units, opposite_distance):
    if side == 'left':
        return (units, distance, opposite_units, opposite_distance)
    return (opposite_units, opposite_distance, units, distance)

def role_record(side, units, distance, opposite_units, opposite_distance):
    shape = shape_for(
        side, units, distance, opposite_units, opposite_distance
    )
    scene_id = scene_by_shape.get(shape)
    if scene_id is None:
        return None
    scene = scene_by_id[scene_id]
    return {
        'scene_id': scene_id,
        'units': int(units),
        'distance': int(distance),
        'side_torque': int(scene['derived'][f'{side}_torque']),
        'net_torque': int(scene['derived']['net_torque']),
        'outcome': str(scene['derived']['outcome']),
    }

def make_candidate(factor, generator, attempt):
    side = SIDES[int(generator.integers(0, len(SIDES)))]
    opposite_configs = [(u, d) for u in range(1, 6) for d in range(1, 7)]
    source_opposite = opposite_configs[
        int(generator.integers(0, len(opposite_configs)))
    ]
    target_choices = [value for value in opposite_configs if value != source_opposite]
    target_opposite = target_choices[
        int(generator.integers(0, len(target_choices)))
    ]
    factor_values = range(1, 6) if factor == 'weight' else range(1, 7)
    base_value, changed_value = generator.choice(
        list(factor_values), size=2, replace=False
    ).tolist()
    context_values = range(1, 7) if factor == 'weight' else range(1, 6)
    source_contexts = tuple(sorted(
        generator.choice(list(context_values), size=3, replace=False).tolist()
    ))
    target_contexts = tuple(sorted(
        generator.choice(list(context_values), size=3, replace=False).tolist()
    ))
    roles = {}
    for key, context in zip(SOURCE_KEYS, source_contexts):
        if factor == 'weight':
            base_units, base_distance = base_value, context
            changed_units, changed_distance = changed_value, context
        else:
            base_units, base_distance = context, base_value
            changed_units, changed_distance = context, changed_value
        roles[f'source_{key}_base'] = role_record(
            side, base_units, base_distance, *source_opposite
        )
        roles[f'source_{key}_changed'] = role_record(
            side, changed_units, changed_distance, *source_opposite
        )
    for key, context in zip(TARGET_KEYS, target_contexts):
        if factor == 'weight':
            base_units, base_distance = base_value, context
            changed_units, changed_distance = changed_value, context
        else:
            base_units, base_distance = context, base_value
            changed_units, changed_distance = context, changed_value
        roles[f'target_{key}_base'] = role_record(
            side, base_units, base_distance, *target_opposite
        )
        roles[f'target_{key}_hybrid'] = role_record(
            side, changed_units, changed_distance, *target_opposite
        )
    if any(value is None for value in roles.values()):
        return None
    scene_ids = [role['scene_id'] for role in roles.values()]
    if len(scene_ids) != len(set(scene_ids)):
        return None
    source_changed_outcomes = {
        roles[f'source_{key}_changed']['outcome'] for key in SOURCE_KEYS
    }
    target_hybrid_outcomes = {
        roles[f'target_{key}_hybrid']['outcome'] for key in TARGET_KEYS
    }
    target_flips = {
        roles[f'target_{key}_base']['outcome']
        != roles[f'target_{key}_hybrid']['outcome']
        for key in TARGET_KEYS
    }
    if len(source_changed_outcomes) < 2:
        return None
    if len(target_hybrid_outcomes) < 2 or target_flips != {False, True}:
        return None
    delta = int(changed_value - base_value)
    signature = (
        factor, side, source_opposite, target_opposite,
        base_value, changed_value, source_contexts, target_contexts,
    )
    family_id = f'equivalence_{factor}_' + stable_key(signature)[:18]
    return {
        'family_id': family_id,
        'factor': factor,
        'changed_side': side,
        'factor_base': int(base_value),
        'factor_changed': int(changed_value),
        'factor_delta': delta,
        'delta_sign': int(np.sign(delta)),
        'source_opposite_units': int(source_opposite[0]),
        'source_opposite_distance': int(source_opposite[1]),
        'target_opposite_units': int(target_opposite[0]),
        'target_opposite_distance': int(target_opposite[1]),
        'source_contexts': dict(zip(SOURCE_KEYS, source_contexts)),
        'target_contexts': dict(zip(TARGET_KEYS, target_contexts)),
        'source_doses': {
            key: abs(delta) * int(context)
            for key, context in zip(SOURCE_KEYS, source_contexts)
        },
        'target_doses': {
            key: abs(delta) * int(context)
            for key, context in zip(TARGET_KEYS, target_contexts)
        },
        'roles': roles,
        'prior_scene_overlap_count': int(sum(
            scene_id in prior_scene_ids for scene_id in scene_ids
        )),
        'source_changed_outcomes': sorted(source_changed_outcomes),
        'target_hybrid_outcomes': sorted(target_hybrid_outcomes),
        'target_flip_pattern': [
            bool(
                roles[f'target_{key}_base']['outcome']
                != roles[f'target_{key}_hybrid']['outcome']
            )
            for key in TARGET_KEYS
        ],
        'generation_attempt': int(attempt),
    }

if CANDIDATE_MANIFEST_PATH.is_file() and CANDIDATE_LOCK_PATH.is_file():
    candidate_manifest = read_jsonl(CANDIDATE_MANIFEST_PATH)
    cell_stage('Using the existing locked candidate manifest')
else:
    generator = np.random.default_rng(SEED)
    pools = defaultdict(dict)
    target_pool_per_group = 300
    max_attempts = 300000
    for attempt in tqdm(
        range(max_attempts), desc='Enumerating valid crossed families', unit='try'
    ):
        factor = FACTORS[attempt % len(FACTORS)]
        row = make_candidate(factor, generator, attempt)
        if row is None:
            continue
        group = (row['factor'], row['changed_side'], row['delta_sign'])
        pools[group][row['family_id']] = row
        required_groups = [
            (factor_name, side, sign)
            for factor_name in FACTORS for side in SIDES for sign in (-1, 1)
        ]
        if all(len(pools[key]) >= target_pool_per_group for key in required_groups):
            break
    required_groups = [
        (factor, side, sign)
        for factor in FACTORS for side in SIDES for sign in (-1, 1)
    ]
    assert all(len(pools[key]) >= target_pool_per_group for key in required_groups), {
        str(key): len(pools[key]) for key in required_groups
    }

    per_group = CANDIDATE_FAMILIES_PER_FACTOR // 4
    assert per_group * 4 == CANDIDATE_FAMILIES_PER_FACTOR
    usage = Counter()
    selected = []
    for round_index in range(per_group):
        for group in required_groups:
            available = [
                row for row in pools[group].values()
                if row['family_id'] not in {item['family_id'] for item in selected}
            ]
            available.sort(key=lambda row: (
                sum(usage[role['scene_id']] for role in row['roles'].values()),
                row['prior_scene_overlap_count'],
                stable_key((round_index, group, row['family_id'])),
            ))
            assert available, group
            chosen = available[0]
            selected.append(chosen)
            usage.update(role['scene_id'] for role in chosen['roles'].values())
    candidate_manifest = sorted(selected, key=lambda row: row['family_id'])
    counterbalance_groups = defaultdict(list)
    for row in candidate_manifest:
        counterbalance_groups[
            (row['factor'], row['changed_side'], row['delta_sign'])
        ].append(row)
    for rows in counterbalance_groups.values():
        rows.sort(key=lambda row: stable_key(('mapping', row['family_id'])))
        for index, row in enumerate(rows):
            row['counterbalance_phase'] = index % len(MAPPING_VARIANTS)
    write_jsonl_atomic(CANDIDATE_MANIFEST_PATH, candidate_manifest)
    write_json_atomic(CANDIDATE_LOCK_PATH, {
        'identity': {
            'schema_version': 'factor-equivalence-candidates-1.0',
            'selected_count': len(candidate_manifest),
            'counts_by_factor': dict(Counter(
                row['factor'] for row in candidate_manifest
            )),
            'counts_by_factor_side_sign': {
                '|'.join(map(str, key)): int(value)
                for key, value in Counter(
                    (row['factor'], row['changed_side'], row['delta_sign'])
                    for row in candidate_manifest
                ).items()
            },
            'selection_uses_model_outputs': False,
            'source_contexts_per_family': len(SOURCE_KEYS),
            'target_contexts_per_family': len(TARGET_KEYS),
            'seed': SEED,
        },
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })

assert len(candidate_manifest) == len(FACTORS) * CANDIDATE_FAMILIES_PER_FACTOR
assert Counter(row['factor'] for row in candidate_manifest) == {
    factor: CANDIDATE_FAMILIES_PER_FACTOR for factor in FACTORS
}
for row in candidate_manifest:
    assert len(set(row['source_contexts'].values())) == len(SOURCE_KEYS)
    assert len(set(row['target_contexts'].values())) == len(TARGET_KEYS)
    assert len({role['scene_id'] for role in row['roles'].values()}) == 12
    assert len(row['source_changed_outcomes']) >= 2
    assert set(row['target_flip_pattern']) == {False, True}

example_config = BehaviorRunConfig(
    dataset_dir=DATASET_DIR,
    output_dir=SHARED_ROOT / '_unused_behavior_loader',
    model_id=MODEL_ID,
    model_revision=MODEL_REVISION,
    conditions=(PromptCondition.IMAGE_RULE,),
    question_kinds=('outcome',),
    splits=('probe_train', 'probe_dev', 'probe_test'),
    batch_size=1, generation_every=0, seed=SEED,
)
outcome_examples = tuple(load_behavior_examples(example_config))
outcome_by_scene = {}
for example in outcome_examples:
    outcome_by_scene.setdefault(str(example.scene_id), example)
selected_scene_ids = {
    str(role['scene_id'])
    for row in candidate_manifest for role in row['roles'].values()
}
missing = selected_scene_ids - set(outcome_by_scene)
assert not missing, sorted(missing)[:5]

candidate_frame = pd.DataFrame([{
    'family_id': row['family_id'],
    'factor': row['factor'],
    'changed_side': row['changed_side'],
    'delta_sign': row['delta_sign'],
    'factor_delta': row['factor_delta'],
    'prior_scene_overlap_count': row['prior_scene_overlap_count'],
    'source_outcomes': '|'.join(row['source_changed_outcomes']),
    'target_outcomes': '|'.join(row['target_hybrid_outcomes']),
} for row in candidate_manifest])
candidate_frame.to_csv(ANALYSIS_DIR / 'candidate_summary.csv', index=False)
display(candidate_frame.groupby([
    'factor', 'changed_side', 'delta_sign'
]).size().rename('n').reset_index())
print('Candidate families / unique scenes:',
      len(candidate_manifest), len(selected_scene_ids))


In [ ]:
%%cell_progress 7/17 Load pinned Qwen-27B and frozen decoder sites
cell_stage('Loading Qwen-27B and resolving L48/L49/L57')
import gc
if 'adapter' in globals():
    del adapter
    gc.collect()
    torch.cuda.empty_cache()

adapter = HuggingFaceMultimodalAdapter(
    MODEL_ID, revision=MODEL_REVISION,
    dtype=MODEL_DTYPE, device_map=DEVICE_MAP,
)
display(adapter.metadata)

def resolve_module_path(root, path):
    value = root
    for component in path.split('.'):
        value = getattr(value, component)
    return value

def discover_decoder_layers(model):
    candidates = (
        'model.language_model.layers', 'model.text_model.layers',
        'language_model.model.layers', 'model.model.layers', 'model.layers',
    )
    for path in candidates:
        try:
            layers = resolve_module_path(model, path)
        except AttributeError:
            continue
        if isinstance(layers, torch.nn.ModuleList) and len(layers) >= 8:
            return path, layers
    raise RuntimeError('Could not locate decoder layers.')

LAYER_PATH, DECODER_LAYERS = discover_decoder_layers(adapter.model)
assert len(DECODER_LAYERS) == EXPECTED_LAYER_COUNT
COMPONENT_MODULES = {
    (layer_index, 'decoder'): DECODER_LAYERS[layer_index]
    for layer_index in CANDIDATE_LAYERS
}
print('Layer path:', LAYER_PATH)
print('Frozen layers:', UPSTREAM_LAYER, CANDIDATE_LAYERS)


## 2. Endpoint competence and crossed factor interventions


In [ ]:
%%cell_progress 8/17 Define counterbalanced prompts and exact token masks
cell_stage('Defining one aligned outcome prompt per family')
import math
from dataclasses import replace as dataclass_replace

RULE_TEXT = (
    'Each square is one unit of weight. Distance is measured in tick '
    'intervals from the center pivot. Torque equals weight times distance. '
    'Total torque on a side is the sum over stacks on that side.'
)
QUESTION_TEXT = 'Which way will the balance scale tip?'
SEMANTIC_TEXT = {
    'left': 'left side', 'balanced': 'equal', 'right': 'right side',
}

def mapping_for(row):
    variant = MAPPING_VARIANTS[int(row['counterbalance_phase']) % 2]
    semantics = (
        ('left', 'balanced', 'right')
        if variant == 'forward'
        else ('right', 'balanced', 'left')
    )
    return variant, tuple(zip(('A', 'B', 'C'), semantics))

def custom_prompt(mapping):
    options = '\n'.join(
        f'{label}. {SEMANTIC_TEXT[semantic]}'
        for label, semantic in mapping
    )
    return '\n\n'.join([
        'Answer this question using only the supplied balance-scale scene.',
        RULE_TEXT,
        'Question: ' + QUESTION_TEXT,
        'Options:\n' + options,
        'Do not explain your reasoning. Reply with exactly one option '
        'letter and nothing else. Valid replies are: A, B, C.',
    ])

def make_example(row, role_name):
    mapping_variant, mapping = mapping_for(row)
    role = row['roles'][role_name]
    scene_id = str(role['scene_id'])
    semantics = tuple(value for _, value in mapping)
    labels = tuple(label for label, _ in mapping)
    outcome = str(role['outcome'])
    base = outcome_by_scene[scene_id]
    return dataclass_replace(
        base,
        example_id=(
            f"factor_equivalence:{row['family_id']}:"
            f"{mapping_variant}:{role_name}"
        ),
        question_id=f"factor_equivalence_{row['family_id']}_{mapping_variant}",
        question_kind='outcome',
        variant='crosscontext_factor_equivalence',
        prompt=custom_prompt(mapping),
        labels=labels,
        option_text=tuple(SEMANTIC_TEXT[value] for value in semantics),
        option_semantics=semantics,
        gold_label=labels[semantics.index(outcome)],
        gold_semantic=outcome,
    )

def make_family_examples(row):
    return {
        role_name: make_example(row, role_name)
        for role_name in row['roles']
    }

def output_hidden_tensor(output):
    if torch.is_tensor(output):
        hidden = output
    elif isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        hidden = output[0]
    elif hasattr(output, 'last_hidden_state') and torch.is_tensor(
        output.last_hidden_state
    ):
        hidden = output.last_hidden_state
    else:
        raise TypeError(f'Unsupported output: {type(output).__name__}')
    assert hidden.ndim == 3
    return hidden

def replace_output_hidden(output, modified):
    if torch.is_tensor(output):
        return modified
    if isinstance(output, tuple):
        return (modified,) + output[1:]
    if isinstance(output, list):
        return [modified] + output[1:]
    raise TypeError(type(output).__name__)

def prepare_examples(examples):
    examples = tuple(examples)
    plan = adapter._candidate_plan(examples[0])
    if plan is None:
        raise RuntimeError('A/B/C labels must be single-token candidates.')
    for example in examples[1:]:
        assert adapter._candidate_plan(example) == plan
    fixed_prefix, token_by_label = plan
    inputs = adapter._append_text_prefix(
        adapter._to_device(adapter._template(
            [example.messages() for example in examples],
            add_generation_prompt=True,
        )),
        fixed_prefix,
    )
    return {
        'examples': examples,
        'inputs': inputs,
        'token_by_label': token_by_label,
    }

def discover_masks(prepared):
    assert len(prepared['examples']) == 1
    example = prepared['examples'][0]
    ids = prepared['inputs']['input_ids'][0].detach().cpu()
    attention = prepared['inputs']['attention_mask'][0].detach().cpu().bool()
    tokenizer = getattr(adapter.processor, 'tokenizer', adapter.processor)
    tokens = tokenizer.convert_ids_to_tokens(ids.tolist())
    image_ids = set()
    if IMAGE_TOKEN_ID_OVERRIDE is not None:
        values = (
            IMAGE_TOKEN_ID_OVERRIDE
            if isinstance(IMAGE_TOKEN_ID_OVERRIDE, (list, tuple))
            else [IMAGE_TOKEN_ID_OVERRIDE]
        )
        image_ids.update(int(value) for value in values)
    for token_id, token_text in zip(ids.tolist(), tokens):
        lower = str(token_text).lower()
        if any(marker in lower for marker in ('image', 'vision', '<img', 'img>')):
            image_ids.add(int(token_id))
    image = torch.tensor(
        [int(token_id) in image_ids for token_id in ids.tolist()],
        dtype=torch.bool,
    ) & attention
    assert bool(image.any()), 'No image tokens detected.'
    final = torch.zeros_like(attention)
    final[int(torch.where(attention)[0][-1])] = True
    all_text = attention & ~image & ~final

    encoding = tokenizer(
        example.prompt, add_special_tokens=False,
        return_offsets_mapping=True,
    )
    prompt_ids = list(encoding['input_ids'])
    offsets = list(encoding['offset_mapping'])
    id_list = ids.tolist()
    starts = [
        start for start in range(len(ids) - len(prompt_ids) + 1)
        if id_list[start:start + len(prompt_ids)] == prompt_ids
    ]
    assert len(starts) == 1, len(starts)
    prompt_start = starts[0]
    sections = example.prompt.split('\n\n')
    assert len(sections) == 5
    names = (
        'intro_tokens', 'rule_tokens', 'question_tokens',
        'options_tokens', 'answer_instruction_tokens',
    )
    masks = {
        'valid_tokens': attention,
        'image_tokens': image,
        'all_text_except_final': all_text,
        'final_prompt_token': final,
    }
    spans = []
    cursor = 0
    for name, section in zip(names, sections):
        start = example.prompt.find(section, cursor)
        end = start + len(section)
        cursor = end
        spans.append((name, start, end))
        masks[name] = torch.zeros_like(attention)
    for offset_index, (token_start, token_end) in enumerate(offsets):
        overlaps = [
            (max(0, min(token_end, end) - max(token_start, start)), name)
            for name, start, end in spans
        ]
        overlap, name = max(overlaps)
        full_index = prompt_start + offset_index
        if overlap > 0 and bool(all_text[full_index]):
            masks[name][full_index] = True
    covered = torch.zeros_like(attention)
    for name in names:
        covered |= masks[name]
    masks['separator_chat_tokens'] = all_text & ~covered
    assert torch.equal(
        masks['options_tokens'] | masks['intro_tokens']
        | masks['rule_tokens'] | masks['question_tokens']
        | masks['answer_instruction_tokens']
        | masks['separator_chat_tokens'],
        all_text,
    )
    masks['tokens'] = tokens
    return masks

def assert_aligned(prepared_by_name):
    names = list(prepared_by_name)
    reference = prepared_by_name[names[0]]
    reference_masks = discover_masks(reference)
    for name in names[1:]:
        current = prepared_by_name[name]
        for key in ('input_ids', 'attention_mask'):
            assert torch.equal(reference['inputs'][key], current['inputs'][key]), (
                name, key
            )
        masks = discover_masks(current)
        for scope in (UPSTREAM_SCOPE, 'final_prompt_token'):
            assert torch.equal(reference_masks[scope], masks[scope]), (
                name, scope
            )
    return reference_masks

def score_records(prepared, logits):
    records = []
    for batch_index, example in enumerate(prepared['examples']):
        logprobs = torch.log_softmax(
            logits[batch_index, -1, :].float(), dim=-1
        )
        scores = [
            float(logprobs[prepared['token_by_label'][label]].detach().cpu())
            for label in example.labels
        ]
        maximum = max(scores)
        weights = [math.exp(value - maximum) for value in scores]
        total = sum(weights)
        predicted_index = int(np.argmax(scores))
        records.append({
            'scene_id': str(example.scene_id),
            'example_id': str(example.example_id),
            'candidate_semantics': list(example.option_semantics),
            'candidate_logprobs': scores,
            'candidate_probabilities': [value / total for value in weights],
            'predicted_semantic': example.option_semantics[predicted_index],
            'gold_semantic': example.gold_semantic,
        })
    return records

def centered_error(first, second):
    first = np.asarray(first, dtype=float)
    second = np.asarray(second, dtype=float)
    return float(np.max(np.abs(
        (first - first.mean()) - (second - second.mean())
    )))

def semantic_margin(record, positive, negative):
    mapping = dict(zip(
        record['candidate_semantics'], record['candidate_logprobs']
    ))
    return float(mapping[positive] - mapping[negative])

def forward_score_prepared(prepared):
    with torch.inference_mode():
        output = adapter.model(**prepared['inputs'], use_cache=False)
    records = score_records(prepared, output.logits)
    assert len(records) == len(prepared['examples'])
    return records

print('Prompt, scoring, and exact token alignment ready.')


In [ ]:
%%cell_progress 9/17 Screen endpoints and freeze the confirmation cohort
cell_stage('Scoring natural endpoints with resumable batch-one requests')

def screen_key(row, role_name):
    mapping_variant, _ = mapping_for(row)
    scene_id = str(row['roles'][role_name]['scene_id'])
    return stable_key(('endpoint', mapping_variant, scene_id))[:24]

endpoint_requests = {}
for row in candidate_manifest:
    for role_name in row['roles']:
        key = screen_key(row, role_name)
        endpoint_requests.setdefault(key, (row, role_name))

progress = tqdm(
    endpoint_requests.items(), desc='Batch-one endpoint competence', unit='image'
)
for key, (row, role_name) in progress:
    path = SCREEN_DIR / f'{key}.json'
    if path.is_file():
        continue
    example = make_example(row, role_name)
    record = forward_score_prepared(prepare_examples([example]))[0]
    record.update({
        'mapping_variant': mapping_for(row)[0],
        'role_name': role_name,
        'correct': bool(record['predicted_semantic'] == record['gold_semantic']),
    })
    write_json_atomic(path, record)
    progress.set_postfix_str(f"{row['factor']} | {role_name}")
progress.close()

screen_cache = {
    path.stem: json.loads(path.read_text(encoding='utf-8'))
    for path in SCREEN_DIR.glob('*.json')
}
screen_rows = []
eligible = []
for row in candidate_manifest:
    role_records = {
        role_name: screen_cache[screen_key(row, role_name)]
        for role_name in row['roles']
    }
    endpoint_correct = sum(
        bool(record['correct']) for record in role_records.values()
    )
    all_correct = endpoint_correct == len(role_records)
    target_contrast_norms = []
    for target_key in TARGET_KEYS:
        base_values = np.asarray(
            role_records[f'target_{target_key}_base']['candidate_logprobs'],
            dtype=float,
        )
        hybrid_values = np.asarray(
            role_records[f'target_{target_key}_hybrid']['candidate_logprobs'],
            dtype=float,
        )
        base_values -= base_values.mean()
        hybrid_values -= hybrid_values.mean()
        target_contrast_norms.append(float(np.linalg.norm(
            hybrid_values - base_values
        )))
    minimum_target_contrast = min(target_contrast_norms)
    observable = minimum_target_contrast >= MIN_TARGET_COUNTERFACTUAL_NORM
    screen_rows.append({
        'family_id': row['family_id'],
        'factor': row['factor'],
        'changed_side': row['changed_side'],
        'delta_sign': row['delta_sign'],
        'correct_endpoints': endpoint_correct,
        'total_endpoints': len(role_records),
        'all_endpoints_correct': all_correct,
        'minimum_target_counterfactual_norm': minimum_target_contrast,
        'all_target_counterfactuals_observable': observable,
        'prior_scene_overlap_count': row['prior_scene_overlap_count'],
    })
    if all_correct and observable:
        eligible.append(row)
screen_frame = pd.DataFrame(screen_rows)
screen_frame.to_csv(ANALYSIS_DIR / 'endpoint_competence_screen.csv', index=False)
display(screen_frame.groupby([
    'factor', 'changed_side', 'delta_sign'
])['all_endpoints_correct'].agg(['sum', 'count']).reset_index())

if CONFIRMATION_MANIFEST_PATH.is_file() and CONFIRMATION_LOCK_PATH.is_file():
    factor_manifest = read_jsonl(CONFIRMATION_MANIFEST_PATH)
    selection_policy = json.loads(
        CONFIRMATION_LOCK_PATH.read_text(encoding='utf-8')
    )['identity']['selection_policy']
    cell_stage('Using the existing locked confirmation cohort')
else:
    per_group = CONFIRMATION_FAMILIES_PER_FACTOR // 4
    assert per_group * 4 == CONFIRMATION_FAMILIES_PER_FACTOR
    required_groups = [
        (factor, side, sign)
        for factor in FACTORS for side in SIDES for sign in (-1, 1)
    ]
    grouped = {
        key: [
            row for row in eligible
            if (row['factor'], row['changed_side'], row['delta_sign']) == key
        ]
        for key in required_groups
    }
    assert all(len(grouped[key]) >= per_group for key in required_groups), {
        str(key): len(grouped[key]) for key in required_groups
    }

    def attempt_selection(attempt, exclude_prior, global_disjoint):
        chosen = []
        used_global = set()
        used_by_factor = {factor: set() for factor in FACTORS}
        mapping_counts = {
            factor: Counter() for factor in FACTORS
        }
        group_order = sorted(
            required_groups, key=lambda key: stable_key((attempt, 'group', key))
        )
        for group in group_order:
            rows = sorted(grouped[group], key=lambda row: (
                row['prior_scene_overlap_count'],
                mapping_counts[row['factor']][mapping_for(row)[0]],
                stable_key((attempt, group, row['family_id'])),
            ))
            group_chosen = 0
            for row in rows:
                scene_ids = {role['scene_id'] for role in row['roles'].values()}
                if exclude_prior and scene_ids & prior_scene_ids:
                    continue
                mapping_variant = mapping_for(row)[0]
                if mapping_counts[row['factor']][mapping_variant] >= (
                    CONFIRMATION_FAMILIES_PER_FACTOR // 2
                ):
                    continue
                forbidden = used_global if global_disjoint else used_by_factor[row['factor']]
                if scene_ids & forbidden:
                    continue
                chosen.append(row)
                used_by_factor[row['factor']].update(scene_ids)
                mapping_counts[row['factor']][mapping_variant] += 1
                if global_disjoint:
                    used_global.update(scene_ids)
                group_chosen += 1
                if group_chosen == per_group:
                    break
            if group_chosen != per_group:
                return None
        if any(
            mapping_counts[factor] != Counter({
                variant: CONFIRMATION_FAMILIES_PER_FACTOR // 2
                for variant in MAPPING_VARIANTS
            })
            for factor in FACTORS
        ):
            return None
        return chosen

    factor_manifest = None
    selection_policy = None
    policies = (
        (True, True, 'global_scene_disjoint_prior_graded_excluded'),
        (False, True, 'global_scene_disjoint_prior_overlap_minimized'),
        (False, False, 'within_factor_scene_disjoint_fallback'),
    )
    for exclude_prior, global_disjoint, label in policies:
        for attempt in range(2048):
            chosen = attempt_selection(attempt, exclude_prior, global_disjoint)
            if chosen is not None:
                factor_manifest = sorted(chosen, key=lambda row: row['family_id'])
                selection_policy = label
                break
        if factor_manifest is not None:
            break
    assert factor_manifest is not None, (
        'Endpoint competence passed, but no balanced scene-disjoint cohort '
        'could be assembled. Increase CANDIDATE_FAMILIES_PER_FACTOR in a new version.'
    )
    write_jsonl_atomic(CONFIRMATION_MANIFEST_PATH, factor_manifest)
    confirmation_scene_ids = [
        role['scene_id']
        for row in factor_manifest for role in row['roles'].values()
    ]
    write_json_atomic(CONFIRMATION_LOCK_PATH, {
        'identity': {
            'schema_version': 'factor-equivalence-confirmation-1.0',
            'selected_count': len(factor_manifest),
            'selection_policy': selection_policy,
            'counts_by_factor_side_sign': {
                '|'.join(map(str, key)): int(value)
                for key, value in Counter(
                    (
                        row['factor'], row['changed_side'], row['delta_sign'],
                        mapping_for(row)[0],
                    )
                    for row in factor_manifest
                ).items()
            },
            'all_natural_endpoints_correct': True,
            'all_target_counterfactuals_observable': True,
            'selection_uses_endpoint_behavior': True,
            'selection_uses_intervention_outputs': False,
            'global_scene_disjoint': len(confirmation_scene_ids) == len(set(confirmation_scene_ids)),
            'family_ids': [row['family_id'] for row in factor_manifest],
            'candidate_manifest_sha256': sha256_file(CANDIDATE_MANIFEST_PATH),
            'seed': SEED,
        },
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })

assert len(factor_manifest) == EXPECTED_FAMILIES
assert Counter(row['factor'] for row in factor_manifest) == {
    factor: CONFIRMATION_FAMILIES_PER_FACTOR for factor in FACTORS
}
assert Counter(
    (row['factor'], row['changed_side'], row['delta_sign'])
    for row in factor_manifest
) == {
    (factor, side, sign): CONFIRMATION_FAMILIES_PER_FACTOR // 4
    for factor in FACTORS for side in SIDES for sign in (-1, 1)
}
assert Counter(
    (row['factor'], mapping_for(row)[0]) for row in factor_manifest
) == {
    (factor, mapping): CONFIRMATION_FAMILIES_PER_FACTOR // 2
    for factor in FACTORS for mapping in MAPPING_VARIANTS
}
for row in factor_manifest:
    assert all(
        screen_cache[screen_key(row, role_name)]['correct']
        for role_name in row['roles']
    )

confirmation_frame = pd.DataFrame([{
    'family_id': row['family_id'],
    'factor': row['factor'],
    'changed_side': row['changed_side'],
    'delta_sign': row['delta_sign'],
    'mapping_variant': mapping_for(row)[0],
    'prior_scene_overlap_count': row['prior_scene_overlap_count'],
} for row in factor_manifest])
confirmation_frame.to_csv(
    ANALYSIS_DIR / 'confirmation_cohort.csv', index=False
)
display(confirmation_frame.groupby([
    'factor', 'changed_side', 'delta_sign', 'mapping_variant'
]).size().rename('n').reset_index())
print('Eligible / frozen families:', len(eligible), len(factor_manifest))
print('Selection policy:', selection_policy)


In [ ]:
%%cell_progress 10/17 Define crossed capture, decomposition, and restoration helpers
cell_stage('Defining common-versus-residual interventions and frozen mediation')

def site_key(layer_index, component='decoder'):
    return f'L{int(layer_index)}:{component}'

def forward_capture(prepared, upstream_payload=None, upstream_mode=None):
    assert len(prepared['examples']) == 1
    masks = discover_masks(prepared)
    upstream_mask = masks[UPSTREAM_SCOPE]
    captured = {}
    handles = []
    if upstream_payload is not None:
        def upstream_hook(_module, _inputs, output):
            hidden = output_hidden_tensor(output)
            selected = upstream_mask.to(hidden.device)
            payload = upstream_payload.to(hidden.device, hidden.dtype)
            modified = hidden.clone()
            if upstream_mode == 'add':
                modified[0, selected] = hidden[0, selected] + payload[selected]
            elif upstream_mode == 'replace':
                modified[0, selected] = payload[selected]
            else:
                raise ValueError(upstream_mode)
            return replace_output_hidden(output, modified)
        handles.append(
            DECODER_LAYERS[UPSTREAM_LAYER].register_forward_hook(upstream_hook)
        )
    def capture_upstream(_module, _inputs, output):
        captured[site_key(UPSTREAM_LAYER)] = (
            output_hidden_tensor(output)[0].detach().cpu().to(torch.bfloat16)
        )
    handles.append(
        DECODER_LAYERS[UPSTREAM_LAYER].register_forward_hook(capture_upstream)
    )
    for (layer_index, component), module in COMPONENT_MODULES.items():
        key = site_key(layer_index, component)
        def capture(_module, _inputs, output, key=key):
            captured[key] = (
                output_hidden_tensor(output)[0].detach().cpu().to(torch.bfloat16)
            )
        handles.append(module.register_forward_hook(capture))
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        for handle in handles:
            handle.remove()
    record = score_records(prepared, output.logits)[0]
    assert len(captured) == 1 + len(COMPONENT_MODULES)
    return record, captured, masks

def forward_score(example):
    return forward_score_prepared(prepare_examples([example]))[0]

def masked_delta(changed, base, mask):
    result = torch.zeros_like(changed, dtype=torch.bfloat16)
    selected = mask.bool()
    result[selected] = (
        changed[selected].float() - base[selected].float()
    ).to(torch.bfloat16)
    return result

def masked_norm(tensor, mask):
    return float(torch.linalg.vector_norm(
        tensor[mask.bool()].float()
    ))

def match_norm(tensor, reference, mask):
    result = torch.zeros_like(tensor, dtype=torch.float32)
    selected = mask.bool()
    values = tensor[selected].float()
    current = torch.linalg.vector_norm(values)
    target = torch.linalg.vector_norm(reference[selected].float())
    if float(current) > 0 and float(target) > 0:
        values = values * (target / current)
    else:
        values.zero_()
    result[selected] = values
    return result.to(torch.bfloat16)

def random_like(reference, mask, key):
    generator = torch.Generator(device='cpu')
    generator.manual_seed(int(stable_key(key)[:16], 16) % (2**63 - 1))
    result = torch.zeros_like(reference, dtype=torch.float32)
    selected = mask.bool()
    values = torch.randn(
        reference[selected].shape, generator=generator, dtype=torch.float32
    )
    target = torch.linalg.vector_norm(reference[selected].float())
    values = values * (target / torch.linalg.vector_norm(values))
    result[selected] = values
    return result.to(torch.bfloat16)

def decompose_source_deltas(deltas, mask):
    stacked = torch.stack([delta.float() for delta in deltas], dim=0)
    shared_float = stacked.mean(dim=0)
    shared = shared_float.to(torch.bfloat16)
    residuals = [
        match_norm(delta.float() - shared_float, shared, mask)
        for delta in deltas
    ]
    raw_residuals = [
        delta.float() - shared_float
        for delta in deltas
    ]
    return shared, residuals, raw_residuals

def score_upstream_conditions(example, conditions):
    assert 1 <= len(conditions) <= SITE_BATCH_SIZE
    prepared = prepare_examples([example] * len(conditions))
    single = prepare_examples([example])
    selected = discover_masks(single)[UPSTREAM_SCOPE]
    def hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        current = selected.to(hidden.device)
        modified = hidden.clone()
        for batch_index, condition in enumerate(conditions):
            payload = condition['payload'].to(hidden.device, hidden.dtype)
            if condition['mode'] == 'add':
                modified[batch_index, current] = (
                    hidden[batch_index, current] + payload[current]
                )
            elif condition['mode'] == 'replace':
                modified[batch_index, current] = payload[current]
            else:
                raise ValueError(condition['mode'])
        return replace_output_hidden(output, modified)
    handle = DECODER_LAYERS[UPSTREAM_LAYER].register_forward_hook(hook)
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        handle.remove()
    records = score_records(prepared, output.logits)
    for record, condition in zip(records, conditions):
        record['condition'] = condition['condition']
        record['upstream_mode'] = condition['mode']
        record['upstream_delta_norm'] = masked_norm(
            condition['payload'], selected
        )
    return records

def masked_removal(additive_state, clean_state, mask):
    removal = torch.zeros_like(additive_state, dtype=torch.bfloat16)
    selected = mask.bool()
    removal[selected] = (
        additive_state[selected].float() - clean_state[selected].float()
    ).to(torch.bfloat16)
    return removal

def score_restoration_conditions(example, upstream_delta, conditions):
    prepared = prepare_examples([example] * len(conditions))
    single = prepare_examples([example])
    masks = discover_masks(single)
    upstream_mask = masks[UPSTREAM_SCOPE]
    handles = []
    measurements = [
        {'restored_norm': 0.0, 'random_norm': 0.0}
        for _ in conditions
    ]
    def upstream_hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        selected = upstream_mask.to(hidden.device)
        payload = upstream_delta.to(hidden.device, hidden.dtype)
        modified = hidden.clone()
        modified[:, selected] = hidden[:, selected] + payload[selected]
        return replace_output_hidden(output, modified)
    handles.append(
        DECODER_LAYERS[UPSTREAM_LAYER].register_forward_hook(upstream_hook)
    )
    union_keys = sorted({
        key for condition in conditions for key in condition['removals']
    })
    for key in union_keys:
        layer_text, component = key.split(':')
        layer_index = int(layer_text[1:])
        module = COMPONENT_MODULES[(layer_index, component)]
        def restore_hook(_module, _inputs, output, key=key):
            hidden = output_hidden_tensor(output)
            modified = hidden.clone()
            for batch_index, condition in enumerate(conditions):
                removal = condition['removals'][key].to(
                    hidden.device, hidden.dtype
                )
                modified[batch_index] = hidden[batch_index] - removal
                norm = float(torch.linalg.vector_norm(removal.float()))
                field = 'random_norm' if condition['random_control'] else 'restored_norm'
                measurements[batch_index][field] += norm ** 2
            return replace_output_hidden(output, modified)
        handles.append(module.register_forward_hook(restore_hook))
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        for handle in handles:
            handle.remove()
    records = score_records(prepared, output.logits)
    for record, condition, measurement in zip(records, conditions, measurements):
        record['condition'] = condition['condition']
        record['upstream_mode'] = 'add_then_restore'
        record['upstream_delta_norm'] = masked_norm(
            upstream_delta, upstream_mask
        )
        record['restored_norm'] = measurement['restored_norm'] ** 0.5
        record['random_norm'] = measurement['random_norm'] ** 0.5
    return records

def counterfactual_metrics(record, clean, natural):
    current = np.asarray(record['candidate_logprobs'], dtype=float)
    base = np.asarray(clean['candidate_logprobs'], dtype=float)
    hybrid = np.asarray(natural['candidate_logprobs'], dtype=float)
    current -= current.mean()
    base -= base.mean()
    hybrid -= hybrid.mean()
    direction = hybrid - base
    denominator = float(direction @ direction)
    if denominator <= 1e-12:
        return float('nan'), float('nan')
    progress = float((current - base) @ direction / denominator)
    ratio = float(
        np.linalg.norm(current - hybrid) / max(np.linalg.norm(direction), 1e-12)
    )
    return progress, ratio

def centered_logprobs(record):
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return values - values.mean()

def source_copy_advantage(record, clean, natural, source_output_delta):
    if source_output_delta is None:
        return None
    current = centered_logprobs(record)
    base = centered_logprobs(clean)
    hybrid = centered_logprobs(natural)
    source_delta = np.asarray(source_output_delta, dtype=float)
    source_prediction = base + source_delta
    target_distance = float(np.linalg.norm(current - hybrid))
    source_distance = float(np.linalg.norm(current - source_prediction))
    scale = max(
        float(np.linalg.norm(hybrid - base)),
        float(np.linalg.norm(source_delta)), 1e-12,
    )
    return float((source_distance - target_distance) / scale)

def chunks(values, size):
    for start in range(0, len(values), size):
        yield values[start:start + size]

print('Crossed factor and frozen-route helpers ready.')


In [ ]:
%%cell_progress 11/17 Run physics, alignment, decomposition, and parity smoke tests
cell_stage('Checking one complete crossed family before long capture')

smoke_row = factor_manifest[0]
smoke_examples = make_family_examples(smoke_row)
smoke_prepared = {
    name: prepare_examples([example])
    for name, example in smoke_examples.items()
}
smoke_masks = assert_aligned(smoke_prepared)
upstream_mask = smoke_masks[UPSTREAM_SCOPE]
assert len(set(smoke_row['source_contexts'].values())) == 3
assert len(set(smoke_row['target_contexts'].values())) == 3
assert len(smoke_row['source_changed_outcomes']) >= 2
assert set(smoke_row['target_flip_pattern']) == {False, True}

smoke_hidden = {}
for source_key in SOURCE_KEYS:
    for suffix in ('base', 'changed'):
        role = f'source_{source_key}_{suffix}'
        _, smoke_hidden[role], _ = forward_capture(smoke_prepared[role])
deltas = [
    masked_delta(
        smoke_hidden[f'source_{key}_changed'][site_key(UPSTREAM_LAYER)],
        smoke_hidden[f'source_{key}_base'][site_key(UPSTREAM_LAYER)],
        upstream_mask,
    )
    for key in SOURCE_KEYS
]
shared, residuals, raw_residuals = decompose_source_deltas(
    deltas, upstream_mask
)
residual_sum = torch.stack(
    [value.float() for value in raw_residuals]
).sum(dim=0)
assert float(torch.max(torch.abs(residual_sum))) < 0.05
random_shared = random_like(shared, upstream_mask, 'smoke:random')
random_norm_error = abs(
    masked_norm(random_shared, upstream_mask)
    - masked_norm(shared, upstream_mask)
) / max(masked_norm(shared, upstream_mask), 1e-12)

target_role = 'target_t1_base'
target_score, target_hidden, _ = forward_capture(
    smoke_prepared[target_role]
)
additive_score, additive_hidden, _ = forward_capture(
    smoke_prepared[target_role], shared, 'add'
)
selected_removals = {}
random_removals = {}
for rank, site in enumerate(FROZEN_SITES[smoke_row['factor']], start=1):
    key = site['site_key']
    mask = smoke_masks[site['scope']]
    removal = masked_removal(
        additive_hidden[key], target_hidden[key], mask
    )
    selected_removals[key] = removal
    random_removals[key] = random_like(
        removal, mask,
        f"smoke:route:{rank}:{key}:{site['scope']}",
    )
restored = score_restoration_conditions(
    smoke_examples[target_role], shared,
    [
        {
            'condition': 'joint_restore',
            'removals': selected_removals,
            'random_control': False,
        },
        {
            'condition': 'joint_random',
            'removals': random_removals,
            'random_control': True,
        },
    ],
)
self_record = score_upstream_conditions(
    smoke_examples[target_role],
    [{
        'condition': 'self_replay',
        'payload': target_hidden[site_key(UPSTREAM_LAYER)],
        'mode': 'replace',
    }],
)[0]
self_error = centered_error(
    self_record['candidate_logprobs'], target_score['candidate_logprobs']
)
route_norm = restored[0]['restored_norm']
route_random_norm = restored[1]['random_norm']
route_norm_error = abs(route_norm - route_random_norm) / max(route_norm, 1e-12)
additive_repeat_error = centered_error(
    additive_score['candidate_logprobs'],
    score_upstream_conditions(
        smoke_examples[target_role],
        [{'condition': 'shared', 'payload': shared, 'mode': 'add'}],
    )[0]['candidate_logprobs'],
)
assert self_error < SMOKE_CENTERED_LOGPROB_TOLERANCE, self_error
assert additive_repeat_error < SMOKE_CENTERED_LOGPROB_TOLERANCE, additive_repeat_error
assert random_norm_error < NORM_RELATIVE_TOLERANCE, random_norm_error
assert route_norm_error < NORM_RELATIVE_TOLERANCE, route_norm_error
print('Smoke family / factor:', smoke_row['family_id'], smoke_row['factor'])
print('Self / additive parity:', self_error, additive_repeat_error)
print('Upstream / route norm errors:', random_norm_error, route_norm_error)


## 3. Frozen confirmation and causal model comparison


In [ ]:
%%cell_progress 12/17 Run crossed source-target interventions
cell_stage('Capturing resumable family shards')

capture_identity = {
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'candidate_manifest_sha256': sha256_file(CANDIDATE_MANIFEST_PATH),
    'confirmation_manifest_sha256': sha256_file(CONFIRMATION_MANIFEST_PATH),
    'model_id': MODEL_ID,
    'revision': MODEL_REVISION,
    'frozen_sites': FROZEN_SITES,
    'conditions': list(CONDITIONS),
}
CAPTURE_SHA256 = 'sha256:' + hashlib.sha256(
    json.dumps(capture_identity, sort_keys=True).encode('utf-8')
).hexdigest()

def add_record_metadata(record, row, target_key, condition, clean, natural,
                        payload_source_key=None, source_output_delta=None):
    changed_side = row['changed_side']
    negative_side = opposite_side(changed_side)
    clean_margin = semantic_margin(clean, changed_side, negative_side)
    current_margin = semantic_margin(record, changed_side, negative_side)
    natural_margin = semantic_margin(natural, changed_side, negative_side)
    direction = int(row['delta_sign'])
    progress, distance_ratio = counterfactual_metrics(record, clean, natural)
    target_context = int(row['target_contexts'][target_key])
    target_hybrid_outcome = str(
        row['roles'][f'target_{target_key}_hybrid']['outcome']
    )
    source_context = None
    source_dose = None
    source_changed_outcome = None
    source_answer_match = None
    if payload_source_key is not None:
        source_context = int(row['source_contexts'][payload_source_key])
        source_dose = float(row['source_doses'][payload_source_key])
        source_changed_outcome = str(
            row['roles'][f'source_{payload_source_key}_changed']['outcome']
        )
        source_answer_match = bool(
            source_changed_outcome == target_hybrid_outcome
        )
    record.update({
        'schema_version': 'factor-equivalence-record-1.0',
        'family_id': row['family_id'],
        'factor': row['factor'],
        'changed_side': changed_side,
        'delta_sign': direction,
        'factor_delta': int(row['factor_delta']),
        'mapping_variant': mapping_for(row)[0],
        'target_key': target_key,
        'target_rank': TARGET_KEYS.index(target_key) + 1,
        'target_context': target_context,
        'target_physical_dose': float(row['target_doses'][target_key]),
        'target_base_outcome': str(
            row['roles'][f'target_{target_key}_base']['outcome']
        ),
        'target_hybrid_outcome': target_hybrid_outcome,
        'condition': condition,
        'payload_source_key': payload_source_key,
        'source_rank': (
            SOURCE_KEYS.index(payload_source_key) + 1
            if payload_source_key is not None else None
        ),
        'source_context': source_context,
        'source_physical_dose': source_dose,
        'source_changed_outcome': source_changed_outcome,
        'source_answer_matches_target': source_answer_match,
        'clean_candidate_logprobs': clean['candidate_logprobs'],
        'natural_candidate_logprobs': natural['candidate_logprobs'],
        'clean_predicted_semantic': clean['predicted_semantic'],
        'natural_predicted_semantic': natural['predicted_semantic'],
        'clean_oriented_margin': direction * clean_margin,
        'natural_oriented_margin': direction * natural_margin,
        'intervention_oriented_margin': direction * current_margin,
        'oriented_effect': direction * (current_margin - clean_margin),
        'natural_oriented_effect': direction * (natural_margin - clean_margin),
        'counterfactual_progress': progress,
        'counterfactual_distance_ratio': distance_ratio,
        'target_over_source_copy_advantage': source_copy_advantage(
            record, clean, natural, source_output_delta
        ),
        'centered_logprob_error_to_clean': centered_error(
            record['candidate_logprobs'], clean['candidate_logprobs']
        ),
    })
    record.setdefault('upstream_mode', 'none')
    record.setdefault('upstream_delta_norm', 0.0)
    record.setdefault('restored_norm', 0.0)
    record.setdefault('random_norm', 0.0)
    return record

progress = tqdm(
    factor_manifest, desc='Crossed factor-equivalence families', unit='family'
)
for row in progress:
    shard_path = SHARD_DIR / f"{row['family_id']}.json"
    if shard_path.is_file():
        existing = json.loads(shard_path.read_text(encoding='utf-8'))
        assert existing['capture_identity_sha256'] == CAPTURE_SHA256
        assert len(existing['records']) == len(TARGET_KEYS) * len(CONDITIONS)
        continue

    examples = make_family_examples(row)
    prepared = {
        role_name: prepare_examples([example])
        for role_name, example in examples.items()
    }
    masks = assert_aligned(prepared)
    upstream_mask = masks[UPSTREAM_SCOPE]

    source_scores = {}
    source_hidden = {}
    deltas = []
    for source_key in SOURCE_KEYS:
        for suffix in ('base', 'changed'):
            role = f'source_{source_key}_{suffix}'
            source_scores[role], source_hidden[role], _ = forward_capture(
                prepared[role]
            )
        deltas.append(masked_delta(
            source_hidden[f'source_{source_key}_changed'][site_key(UPSTREAM_LAYER)],
            source_hidden[f'source_{source_key}_base'][site_key(UPSTREAM_LAYER)],
            upstream_mask,
        ))
        torch.cuda.empty_cache()
    shared, residuals, _ = decompose_source_deltas(deltas, upstream_mask)
    source_output_deltas = {}
    for source_key in SOURCE_KEYS:
        source_output_deltas[source_key] = (
            centered_logprobs(source_scores[f'source_{source_key}_changed'])
            - centered_logprobs(source_scores[f'source_{source_key}_base'])
        )
    shared_source_output_delta = np.mean(
        np.stack(list(source_output_deltas.values()), axis=0), axis=0
    )
    random_shared = random_like(
        shared, upstream_mask, f"family:{row['family_id']}:random_shared"
    )

    family_records = []
    for target_key in TARGET_KEYS:
        base_role = f'target_{target_key}_base'
        hybrid_role = f'target_{target_key}_hybrid'
        clean, target_hidden, _ = forward_capture(prepared[base_role])
        natural = forward_score(examples[hybrid_role])

        upstream_conditions = []
        for index, source_key in enumerate(SOURCE_KEYS):
            upstream_conditions.append({
                'condition': f'raw_source_{source_key}',
                'payload': deltas[index], 'mode': 'add',
                'source_key': source_key,
            })
        upstream_conditions.append({
            'condition': 'shared', 'payload': shared,
            'mode': 'add', 'source_key': None,
        })
        for index, source_key in enumerate(SOURCE_KEYS):
            upstream_conditions.append({
                'condition': f'residual_{source_key}',
                'payload': residuals[index], 'mode': 'add',
                'source_key': source_key,
            })
        upstream_conditions.extend([
            {
                'condition': 'reverse_shared', 'payload': -shared,
                'mode': 'add', 'source_key': None,
            },
            {
                'condition': 'random_shared', 'payload': random_shared,
                'mode': 'add', 'source_key': None,
            },
        ])
        for source_key in SOURCE_KEYS:
            upstream_conditions.append({
                'condition': f'whole_source_{source_key}',
                'payload': source_hidden[
                    f'source_{source_key}_changed'
                ][site_key(UPSTREAM_LAYER)],
                'mode': 'replace', 'source_key': source_key,
            })
        upstream_conditions.append({
            'condition': 'self_replay',
            'payload': target_hidden[site_key(UPSTREAM_LAYER)],
            'mode': 'replace', 'source_key': None,
        })

        intervention_records = []
        for batch in chunks(upstream_conditions, SITE_BATCH_SIZE):
            current = score_upstream_conditions(examples[base_role], batch)
            for record, condition in zip(current, batch):
                record['_source_key'] = condition['source_key']
            intervention_records.extend(current)

        shared_score, additive_hidden, _ = forward_capture(
            prepared[base_role], shared, 'add'
        )
        repeated_shared = next(
            record for record in intervention_records
            if record['condition'] == 'shared'
        )
        assert centered_error(
            shared_score['candidate_logprobs'],
            repeated_shared['candidate_logprobs'],
        ) < SMOKE_CENTERED_LOGPROB_TOLERANCE

        selected_removals = {}
        random_removals = {}
        for rank, site in enumerate(FROZEN_SITES[row['factor']], start=1):
            key = site['site_key']
            mask = masks[site['scope']]
            removal = masked_removal(
                additive_hidden[key], target_hidden[key], mask
            )
            selected_removals[key] = removal
            random_removals[key] = random_like(
                removal, mask,
                f"route:{row['family_id']}:{target_key}:"
                f"{rank}:{key}:{site['scope']}",
            )
        route_records = score_restoration_conditions(
            examples[base_role], shared,
            [
                {
                    'condition': 'joint_restore',
                    'removals': selected_removals,
                    'random_control': False,
                },
                {
                    'condition': 'joint_random',
                    'removals': random_removals,
                    'random_control': True,
                },
            ],
        )

        clean_record = dict(clean)
        clean_record['condition'] = 'clean'
        natural_record = dict(natural)
        natural_record['condition'] = 'natural_hybrid'
        cell_records = [clean_record, natural_record]
        cell_records.extend(intervention_records)
        cell_records.extend(route_records)
        assert {record['condition'] for record in cell_records} == set(CONDITIONS)
        for record in cell_records:
            source_key = record.pop('_source_key', None)
            source_output_delta = None
            if record['condition'] in RAW_CONDITIONS + WHOLE_CONDITIONS:
                source_output_delta = source_output_deltas[source_key]
            elif record['condition'] in (
                'shared', 'joint_restore', 'joint_random'
            ):
                source_output_delta = shared_source_output_delta
            add_record_metadata(
                record, row, target_key, record['condition'], clean, natural,
                payload_source_key=source_key,
                source_output_delta=source_output_delta,
            )
            family_records.append(record)
        del target_hidden, additive_hidden
        torch.cuda.empty_cache()

    assert len(family_records) == len(TARGET_KEYS) * len(CONDITIONS)
    write_json_atomic(shard_path, {
        'schema_version': 'factor-equivalence-family-shard-1.0',
        'capture_identity_sha256': CAPTURE_SHA256,
        'family': row,
        'records': family_records,
    })
    progress.set_postfix_str(f"{row['factor']} | {row['family_id']}")
    del source_hidden, source_scores, deltas, shared, residuals
    torch.cuda.empty_cache()
progress.close()

shard_paths = sorted(SHARD_DIR.glob('*.json'))
assert len(shard_paths) == EXPECTED_FAMILIES
write_json_atomic(SHARED_ROOT / 'capture_complete.json', {
    'capture_identity': capture_identity,
    'capture_identity_sha256': CAPTURE_SHA256,
    'families': EXPECTED_FAMILIES,
    'records': EXPECTED_FAMILIES * len(TARGET_KEYS) * len(CONDITIONS),
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
})
print('Complete family shards:', len(shard_paths))


In [ ]:
%%cell_progress 13/17 Assemble crossed family estimands
cell_stage('Releasing GPU and building family-level causal summaries')
import gc
try:
    del adapter
    del DECODER_LAYERS
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

records = []
for path in sorted(SHARD_DIR.glob('*.json')):
    document = json.loads(path.read_text(encoding='utf-8'))
    assert document['capture_identity_sha256'] == CAPTURE_SHA256
    records.extend(document['records'])
record_frame = pd.DataFrame(records)
assert len(record_frame) == EXPECTED_FAMILIES * len(TARGET_KEYS) * len(CONDITIONS)
record_frame.to_csv(ANALYSIS_DIR / 'factor_equivalence_records.csv', index=False)

raw_frame = record_frame[record_frame['condition'].isin(RAW_CONDITIONS)].copy()
raw_frame['source_answer_matches_target'] = (
    raw_frame['source_answer_matches_target'].astype(float)
)
assert len(raw_frame) == EXPECTED_FAMILIES * len(TARGET_KEYS) * len(SOURCE_KEYS)

def simple_slope(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    centered = x - x.mean()
    denominator = float(np.sum(centered ** 2))
    assert denominator > 0
    return float(np.sum(centered * (y - y.mean())) / denominator)

def standardized_crossed_coefficients(frame):
    names = (
        'target_physical_dose', 'source_physical_dose',
        'source_answer_matches_target',
    )
    columns = []
    for name in names:
        values = frame[name].to_numpy(dtype=float)
        scale = float(values.std())
        columns.append(
            (values - values.mean()) / scale
            if scale > 0 else np.zeros_like(values)
        )
    X = np.column_stack([np.ones(len(frame)), *columns])
    y = frame['oriented_effect'].to_numpy(dtype=float)
    coefficients = np.linalg.lstsq(X, y, rcond=None)[0]
    return {
        'target_beta': float(coefficients[1]),
        'source_beta': float(coefficients[2]),
        'source_answer_beta': float(coefficients[3]),
    }

def progress_nuisance_coefficients(frame):
    target_rank = frame['target_rank'].to_numpy(dtype=int)
    target_two = (target_rank == 2).astype(float)
    target_three = (target_rank == 3).astype(float)
    source_dose = frame['source_physical_dose'].to_numpy(dtype=float)
    answer_match = frame['source_answer_matches_target'].to_numpy(dtype=float)
    source_scale = float(source_dose.std())
    answer_scale = float(answer_match.std())
    source_standard = (
        (source_dose - source_dose.mean()) / source_scale
        if source_scale > 0 else np.zeros_like(source_dose)
    )
    answer_standard = (
        (answer_match - answer_match.mean()) / answer_scale
        if answer_scale > 0 else np.zeros_like(answer_match)
    )
    X = np.column_stack([
        np.ones(len(frame)), target_two, target_three,
        source_standard, answer_standard,
    ])
    y = frame['counterfactual_progress'].to_numpy(dtype=float)
    coefficients = np.linalg.lstsq(X, y, rcond=None)[0]
    return {
        'progress_source_beta': float(coefficients[-2]),
        'progress_answer_beta': float(coefficients[-1]),
    }

family_rows = []
for (factor, family_id), current in record_frame.groupby(['factor', 'family_id']):
    row = {'factor': factor, 'family_id': family_id}
    for condition in (
        'shared', 'random_shared', 'reverse_shared',
        'joint_restore', 'joint_random', 'natural_hybrid',
    ):
        values = current[current['condition'] == condition].sort_values(
            'target_physical_dose'
        )
        row[f'{condition}_slope'] = simple_slope(
            values['target_physical_dose'], values['oriented_effect']
        )
        row[f'{condition}_progress'] = float(
            values['counterfactual_progress'].mean()
        )
        row[f'{condition}_distance_ratio'] = float(
            values['counterfactual_distance_ratio'].mean()
        )

    raw_current = raw_frame[
        (raw_frame['factor'] == factor)
        & (raw_frame['family_id'] == family_id)
    ]
    coefficients = standardized_crossed_coefficients(raw_current)
    row.update(coefficients)
    row.update(progress_nuisance_coefficients(raw_current))
    row['raw_progress_mean'] = float(
        raw_current['counterfactual_progress'].mean()
    )
    raw_source_slopes = []
    raw_spreads = []
    for source_key in SOURCE_KEYS:
        values = raw_current[
            raw_current['payload_source_key'] == source_key
        ].sort_values('target_physical_dose')
        raw_source_slopes.append(simple_slope(
            values['target_physical_dose'], values['oriented_effect']
        ))
    for _, values in raw_current.groupby('target_key'):
        raw_spreads.append(float(values['oriented_effect'].std(ddof=0)))
    row['raw_target_slope'] = float(np.mean(raw_source_slopes))
    row['raw_source_spread'] = float(np.mean(raw_spreads))
    row['raw_progress_source_spread'] = float(np.mean([
        values['counterfactual_progress'].std(ddof=0)
        for _, values in raw_current.groupby('target_key')
    ]))

    residual_slopes = []
    residual_progress = []
    for condition in RESIDUAL_CONDITIONS:
        values = current[current['condition'] == condition].sort_values(
            'target_physical_dose'
        )
        residual_slopes.append(simple_slope(
            values['target_physical_dose'], values['oriented_effect']
        ))
        residual_progress.append(float(values['counterfactual_progress'].mean()))
    row['residual_mean_slope'] = float(np.mean(residual_slopes))
    row['residual_mean_progress'] = float(np.mean(residual_progress))
    row['shared_vs_residual_slope'] = (
        row['shared_slope'] - row['residual_mean_slope']
    )
    row['shared_vs_residual_progress'] = (
        row['shared_progress'] - row['residual_mean_progress']
    )
    row['shared_vs_random_slope'] = (
        row['shared_slope'] - row['random_shared_slope']
    )
    row['shared_vs_random_progress'] = (
        row['shared_progress'] - row['random_shared_progress']
    )
    row['route_slope_attenuation'] = (
        row['shared_slope'] - row['joint_restore_slope']
    )
    row['route_slope_specificity'] = (
        row['joint_random_slope'] - row['joint_restore_slope']
    )
    row['route_progress_attenuation'] = (
        row['shared_progress'] - row['joint_restore_progress']
    )
    row['route_progress_specificity'] = (
        row['joint_random_progress'] - row['joint_restore_progress']
    )
    row['shared_progress_advantage_over_random'] = (
        row['shared_progress'] - row['random_shared_progress']
    )
    row['shared_distance_advantage_over_random'] = (
        row['random_shared_distance_ratio'] - row['shared_distance_ratio']
    )
    row['shared_target_over_source_copy_advantage'] = float(
        current[current['condition'] == 'shared'][
            'target_over_source_copy_advantage'
        ].mean()
    )
    row['raw_target_over_source_copy_advantage'] = float(
        raw_current['target_over_source_copy_advantage'].mean()
    )
    family_rows.append(row)

family_frame = pd.DataFrame(family_rows)
assert len(family_frame) == EXPECTED_FAMILIES
family_frame.to_csv(ANALYSIS_DIR / 'family_estimands.csv', index=False)
display(family_frame.groupby('factor').mean(numeric_only=True).round(5))


In [ ]:
%%cell_progress 14/17 Test factor equivalence, common components, and route mediation
cell_stage('Bootstrapping independent families against frozen alternatives')

METRICS = (
    'shared_slope', 'random_shared_slope', 'joint_restore_slope',
    'joint_random_slope', 'raw_target_slope', 'target_beta', 'source_beta',
    'source_answer_beta', 'progress_source_beta', 'progress_answer_beta',
    'raw_progress_mean', 'raw_progress_source_spread',
    'raw_source_spread', 'residual_mean_slope',
    'residual_mean_progress', 'shared_vs_residual_progress',
    'shared_vs_random_progress', 'joint_restore_progress',
    'joint_random_progress', 'route_progress_attenuation',
    'route_progress_specificity',
    'shared_vs_residual_slope', 'shared_vs_random_slope',
    'route_slope_attenuation', 'route_slope_specificity',
    'shared_progress', 'random_shared_progress',
    'shared_progress_advantage_over_random',
    'shared_distance_advantage_over_random', 'natural_hybrid_slope',
    'shared_target_over_source_copy_advantage',
    'raw_target_over_source_copy_advantage',
)

def bootstrap_factor(frame, factor):
    current = frame[frame['factor'] == factor].reset_index(drop=True)
    assert len(current) == CONFIRMATION_FAMILIES_PER_FACTOR
    generator = np.random.default_rng(
        int(stable_key(f'bootstrap:{factor}')[:16], 16) % (2**63 - 1)
    )
    draws = generator.integers(
        0, len(current), size=(BOOTSTRAP_REPLICATES, len(current))
    )
    output = {}
    metric_draws = {}
    for metric in METRICS:
        values = current[metric].to_numpy(dtype=float)
        estimates = values[draws].mean(axis=1)
        metric_draws[metric] = estimates
        output[metric] = {
            'estimate': float(values.mean()),
            'ci_low': float(np.quantile(estimates, ALPHA / 2)),
            'ci_high': float(np.quantile(estimates, 1 - ALPHA / 2)),
            'n_families': int(len(values)),
        }
    progress_draws = metric_draws['raw_progress_mean']
    source_ratio = np.abs(metric_draws['progress_source_beta']) / np.maximum(
        np.abs(progress_draws), 1e-8
    )
    answer_ratio = np.abs(metric_draws['progress_answer_beta']) / np.maximum(
        np.abs(progress_draws), 1e-8
    )
    for name, estimates in (
        ('source_nuisance_to_progress_ratio', source_ratio),
        ('answer_nuisance_to_progress_ratio', answer_ratio),
    ):
        output[name] = {
            'estimate': float(np.median(estimates)),
            'ci_low': float(np.quantile(estimates, ALPHA / 2)),
            'ci_high': float(np.quantile(estimates, 1 - ALPHA / 2)),
            'n_families': int(len(current)),
        }
    return output

test_rows = []
decisions = {}
for factor in FACTORS:
    summaries = bootstrap_factor(family_frame, factor)
    for metric, summary in summaries.items():
        test_rows.append({'factor': factor, 'metric': metric, **summary})
    gates = {
        'shared_matches_natural_composite': (
            summaries['shared_progress']['ci_low'] > 0
        ),
        'raw_matches_target_composites': (
            summaries['raw_progress_mean']['ci_low'] > 0
        ),
        'source_context_equivalent': (
            summaries['source_nuisance_to_progress_ratio']['ci_high']
            < SOURCE_EQUIVALENCE_RATIO
        ),
        'source_answer_equivalent': (
            summaries['answer_nuisance_to_progress_ratio']['ci_high']
            < ANSWER_EQUIVALENCE_RATIO
        ),
        'shared_beats_source_specific_residual': (
            summaries['shared_vs_residual_progress']['ci_low'] > 0
        ),
        'shared_beats_equal_norm_random': (
            summaries['shared_vs_random_progress']['ci_low'] > 0
        ),
        'moves_toward_natural_counterfactual': (
            summaries['shared_progress_advantage_over_random']['ci_low'] > 0
            and summaries['shared_distance_advantage_over_random']['ci_low'] > 0
        ),
        'raw_closer_to_target_than_source_copy': (
            summaries['raw_target_over_source_copy_advantage']['ci_low'] > 0
        ),
        'route_attenuates_shared_counterfactual_progress': (
            summaries['route_progress_attenuation']['ci_low'] > 0
        ),
        'route_specific_over_random': (
            summaries['route_progress_specificity']['ci_low'] > 0
        ),
    }
    factor_equivalence = all(gates[key] for key in (
        'shared_matches_natural_composite', 'raw_matches_target_composites',
        'source_context_equivalent', 'source_answer_equivalent',
        'shared_beats_source_specific_residual',
        'shared_beats_equal_norm_random',
        'moves_toward_natural_counterfactual',
        'raw_closer_to_target_than_source_copy',
    ))
    decisions[factor] = {
        'factor_equivalence_confirmed': bool(factor_equivalence),
        'route_mediation_confirmed': bool(
            gates['route_attenuates_shared_counterfactual_progress']
            and gates['route_specific_over_random']
        ),
        'gates': {key: bool(value) for key, value in gates.items()},
        'summaries': summaries,
    }

if all(
    value['factor_equivalence_confirmed']
    and value['route_mediation_confirmed']
    for value in decisions.values()
):
    PRIMARY_DECISION = 'context_general_factor_equivalence_confirmed_for_both_factors'
elif any(
    value['factor_equivalence_confirmed']
    for value in decisions.values()
):
    PRIMARY_DECISION = 'context_general_factor_equivalence_confirmed_for_one_factor'
elif any(
    value['gates']['shared_matches_natural_composite']
    and not value['gates']['source_context_equivalent']
    for value in decisions.values()
):
    PRIMARY_DECISION = 'target_recombination_with_causally_mixed_source_context'
elif any(
    value['gates']['shared_beats_source_specific_residual']
    for value in decisions.values()
):
    PRIMARY_DECISION = 'shared_factor_effect_without_confirmed_product_interaction'
else:
    PRIMARY_DECISION = 'no_confirmatory_context_general_factor_equivalence'

test_frame = pd.DataFrame(test_rows)
test_frame.to_csv(ANALYSIS_DIR / 'factor_equivalence_tests.csv', index=False)
decision_document = {
    'schema_version': 'factor-equivalence-decision-1.0',
    'primary_decision': PRIMARY_DECISION,
    'factor_decisions': decisions,
    'interpretation_guardrail': (
        'Passing supports a context-general and causally reusable factor '
        'signal. It does not prove statistically independent neurons or '
        'the absence of all decodable source information.'
    ),
}
write_json_atomic(
    ANALYSIS_DIR / 'factor_equivalence_decision.json', decision_document
)
display(test_frame.round(6))
print('Primary decision:', PRIMARY_DECISION)


In [ ]:
%%cell_progress 15/17 Audit support, physics, screening, norms, and replay
cell_stage('Verifying that the result is interpretable under the locked protocol')

confirmation_lock = json.loads(
    CONFIRMATION_LOCK_PATH.read_text(encoding='utf-8')
)['identity']
physics_violations = []
confirmation_scene_ids = []
for row in factor_manifest:
    confirmation_scene_ids.extend(
        role['scene_id'] for role in row['roles'].values()
    )
    for source_key in SOURCE_KEYS:
        base = row['roles'][f'source_{source_key}_base']
        changed = row['roles'][f'source_{source_key}_changed']
        if row['factor'] == 'weight':
            valid = (
                base['distance'] == changed['distance']
                and base['units'] != changed['units']
            )
        else:
            valid = (
                base['units'] == changed['units']
                and base['distance'] != changed['distance']
            )
        if not valid:
            physics_violations.append((row['family_id'], 'source', source_key))
    for target_key in TARGET_KEYS:
        base = row['roles'][f'target_{target_key}_base']
        hybrid = row['roles'][f'target_{target_key}_hybrid']
        if row['factor'] == 'weight':
            valid = (
                base['distance'] == hybrid['distance']
                and base['units'] != hybrid['units']
            )
        else:
            valid = (
                base['units'] == hybrid['units']
                and base['distance'] != hybrid['distance']
            )
        if not valid:
            physics_violations.append((row['family_id'], 'target', target_key))

self_rows = record_frame[record_frame['condition'] == 'self_replay']
random_rows = record_frame[record_frame['condition'] == 'random_shared']
shared_rows = record_frame[record_frame['condition'] == 'shared']
route_rows = record_frame[
    record_frame['condition'].isin(['joint_restore', 'joint_random'])
]
random_norms = random_rows.set_index(['family_id', 'target_key'])[
    'upstream_delta_norm'
]
shared_norms = shared_rows.set_index(['family_id', 'target_key'])[
    'upstream_delta_norm'
]
upstream_norm_error = float(np.max(np.abs(
    random_norms.sort_index().to_numpy()
    - shared_norms.sort_index().to_numpy()
) / np.maximum(shared_norms.sort_index().to_numpy(), 1e-12)))
route_pivot = route_rows.pivot_table(
    index=['family_id', 'target_key'], columns='condition',
    values=['restored_norm', 'random_norm'], aggfunc='first',
)
route_real = route_pivot[('restored_norm', 'joint_restore')].to_numpy()
route_random = route_pivot[('random_norm', 'joint_random')].to_numpy()
route_norm_error = float(np.max(
    np.abs(route_real - route_random) / np.maximum(route_real, 1e-12)
))

numeric_columns = (
    'oriented_effect', 'natural_oriented_effect',
    'counterfactual_progress', 'counterfactual_distance_ratio',
    'upstream_delta_norm',
)
nonfinite_count = int(sum(
    (~np.isfinite(record_frame[column].to_numpy(dtype=float))).sum()
    for column in numeric_columns
))
audit = {
    'schema_version': 'factor-equivalence-audit-1.0',
    'families': int(len(factor_manifest)),
    'records': int(len(record_frame)),
    'counts_by_factor': dict(Counter(
        row['factor'] for row in factor_manifest
    )),
    'counts_by_factor_side_sign': {
        '|'.join(map(str, key)): int(value)
        for key, value in Counter(
            (row['factor'], row['changed_side'], row['delta_sign'])
            for row in factor_manifest
        ).items()
    },
    'mapping_counts': dict(Counter(
        mapping_for(row)[0] for row in factor_manifest
    )),
    'selection_policy': confirmation_lock['selection_policy'],
    'candidate_selection_uses_model_outputs': False,
    'confirmation_uses_endpoint_behavior_only': True,
    'selection_uses_intervention_outputs': False,
    'all_natural_endpoints_correct': bool(
        confirmation_lock['all_natural_endpoints_correct']
    ),
    'all_target_counterfactuals_observable': bool(
        confirmation_lock['all_target_counterfactuals_observable']
    ),
    'minimum_target_counterfactual_norm_threshold': MIN_TARGET_COUNTERFACTUAL_NORM,
    'global_scene_disjoint': bool(
        len(confirmation_scene_ids) == len(set(confirmation_scene_ids))
    ),
    'unique_confirmation_scenes': int(len(set(confirmation_scene_ids))),
    'prior_graded_scene_overlap_count': int(sum(
        scene_id in prior_scene_ids for scene_id in set(confirmation_scene_ids)
    )),
    'physics_violations': int(len(physics_violations)),
    'maximum_self_replay_centered_error': float(
        self_rows['centered_logprob_error_to_clean'].max()
    ),
    'maximum_upstream_equal_norm_relative_error': upstream_norm_error,
    'maximum_route_equal_norm_relative_error': route_norm_error,
    'nonfinite_count': nonfinite_count,
    'source_contexts_per_family': len(SOURCE_KEYS),
    'target_contexts_per_family': len(TARGET_KEYS),
    'source_answers_vary_within_every_family': bool(all(
        len(row['source_changed_outcomes']) >= 2 for row in factor_manifest
    )),
    'target_contains_flip_and_no_flip_within_every_family': bool(all(
        set(row['target_flip_pattern']) == {False, True}
        for row in factor_manifest
    )),
}
assert audit['physics_violations'] == 0, physics_violations[:5]
assert audit['maximum_self_replay_centered_error'] < SMOKE_CENTERED_LOGPROB_TOLERANCE
assert upstream_norm_error < NORM_RELATIVE_TOLERANCE
assert route_norm_error < NORM_RELATIVE_TOLERANCE
assert nonfinite_count == 0
write_json_atomic(ANALYSIS_DIR / 'factor_equivalence_audit.json', audit)
display(pd.DataFrame([
    {'item': key, 'value': str(value)} for key, value in audit.items()
]))


In [ ]:
%%cell_progress 16/17 Save the paper-facing crossed-mechanism figure
cell_stage('Rendering transfer matrices, coefficients, controls, and mediation')
import matplotlib.pyplot as plt

colors = {
    'weight': '#2563EB',
    'distance': '#D97706',
    'control': '#6B7280',
}
fig, axes = plt.subplots(2, 3, figsize=(17, 10))

for axis, factor in zip(axes[0, :2], FACTORS):
    matrix = raw_frame[raw_frame['factor'] == factor].pivot_table(
        index='source_rank', columns='target_rank',
        values='oriented_effect', aggfunc='mean',
    ).reindex(index=(1, 2, 3), columns=(1, 2, 3))
    image = axis.imshow(matrix.to_numpy(), cmap='RdBu_r', aspect='auto')
    axis.set_xticks(range(3), ['target low', 'target mid', 'target high'])
    axis.set_yticks(range(3), ['source low', 'source mid', 'source high'])
    axis.set_title(f'{factor.title()}: raw source × target effect')
    fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)

def summary(factor, metric):
    row = test_frame[
        (test_frame['factor'] == factor) & (test_frame['metric'] == metric)
    ].iloc[0]
    return float(row['estimate']), float(row['ci_low']), float(row['ci_high'])

slope_metrics = (
    ('shared_slope', 'Shared'),
    ('residual_mean_slope', 'Residual'),
    ('random_shared_slope', 'Random'),
    ('natural_hybrid_slope', 'Natural'),
)
x = np.arange(len(slope_metrics))
width = 0.34
for offset, factor in enumerate(FACTORS):
    values, lower, upper = [], [], []
    for metric, _ in slope_metrics:
        estimate, low, high = summary(factor, metric)
        values.append(estimate)
        lower.append(estimate - low)
        upper.append(high - estimate)
    axes[0, 2].bar(
        x + (offset - 0.5) * width, values, width,
        yerr=np.vstack([lower, upper]), capsize=3,
        color=colors[factor], label=factor,
    )
axes[0, 2].axhline(0, color='black', linewidth=0.8)
axes[0, 2].set_xticks(x, [label for _, label in slope_metrics])
axes[0, 2].set_ylabel('Target physical-dose slope')
axes[0, 2].set_title('Common component versus controls')
axes[0, 2].legend(frameon=False)

coefficient_metrics = (
    ('target_beta', 'Target context'),
    ('source_beta', 'Source context'),
    ('source_answer_beta', 'Source answer'),
)
x = np.arange(len(coefficient_metrics))
for offset, factor in enumerate(FACTORS):
    values, lower, upper = [], [], []
    for metric, _ in coefficient_metrics:
        estimate, low, high = summary(factor, metric)
        values.append(estimate)
        lower.append(estimate - low)
        upper.append(high - estimate)
    axes[1, 0].bar(
        x + (offset - 0.5) * width, values, width,
        yerr=np.vstack([lower, upper]), capsize=3,
        color=colors[factor], label=factor,
    )
axes[1, 0].axhline(0, color='black', linewidth=0.8)
axes[1, 0].set_xticks(x, [label for _, label in coefficient_metrics])
axes[1, 0].set_ylabel('Standardized crossed coefficient')
axes[1, 0].set_title('Does the effect follow target or source?')

progress_metrics = (
    ('shared_progress', 'Shared'),
    ('residual_mean_progress', 'Residual'),
    ('random_shared_progress', 'Random'),
)
x = np.arange(len(progress_metrics))
for offset, factor in enumerate(FACTORS):
    values = [
        float(family_frame[
            family_frame['factor'] == factor
        ][metric].mean())
        for metric, _ in progress_metrics
    ]
    axes[1, 1].bar(
        x + (offset - 0.5) * width, values, width,
        color=colors[factor], label=factor,
    )
axes[1, 1].axhline(0, color='black', linewidth=0.8)
axes[1, 1].set_xticks(x, [label for _, label in progress_metrics])
axes[1, 1].set_ylabel('Progress toward natural counterfactual')
axes[1, 1].set_title('Distribution-level counterfactual match')

route_metrics = (
    ('shared_progress', 'Shared'),
    ('joint_restore_progress', 'Route restored'),
    ('joint_random_progress', 'Random restored'),
)
x = np.arange(len(route_metrics))
for offset, factor in enumerate(FACTORS):
    values, lower, upper = [], [], []
    for metric, _ in route_metrics:
        estimate, low, high = summary(factor, metric)
        values.append(estimate)
        lower.append(estimate - low)
        upper.append(high - estimate)
    axes[1, 2].bar(
        x + (offset - 0.5) * width, values, width,
        yerr=np.vstack([lower, upper]), capsize=3,
        color=colors[factor], label=factor,
    )
axes[1, 2].axhline(0, color='black', linewidth=0.8)
axes[1, 2].set_xticks(x, [label for _, label in route_metrics])
axes[1, 2].set_ylabel('Progress toward natural counterfactual')
axes[1, 2].set_title('Frozen-route mediation')

fig.suptitle(
    'Qwen-27B: Cross-Context Factor Equivalence and Product Interaction',
    fontsize=16,
)
fig.tight_layout(rect=(0, 0, 1, 0.96))
figure_path = FIGURE_DIR / 'factor_equivalence_product_interaction.png'
fig.savefig(figure_path, dpi=220, bbox_inches='tight')
plt.show()
print('Figure:', figure_path)


In [ ]:
%%cell_progress 17/17 Write the confirmatory decision report
cell_stage('Saving paper tables, plain-language interpretation, and completion lock')

def interval(factor, metric):
    row = test_frame[
        (test_frame['factor'] == factor) & (test_frame['metric'] == metric)
    ].iloc[0]
    return f"{row['estimate']:.5f} [{row['ci_low']:.5f}, {row['ci_high']:.5f}]"

lines = [
    '# Cross-Context Factor Equivalence and Product Interaction', '',
    f'**Primary decision:** `{PRIMARY_DECISION}`', '',
    '## What was tested', '',
    (
        'Each physical factor change was obtained from three source scenes '
        'with different complementary factors and different outcomes, then '
        'crossed with three target scenes that retained different values of '
        'the complementary factor. The same design was run for weight and '
        'distance. All twelve natural endpoints in every selected family '
        'were solved correctly before any intervention result was inspected.'
    ), '',
    (
        'A factor-like signal must follow the complementary value in the '
        'target scene while being approximately unaffected by the source '
        'scene complement and source answer. The component common across '
        'the three source changes must also outperform source-specific '
        'residuals and an equal-norm random direction.'
    ), '',
    '## Results', '',
]
for factor in FACTORS:
    current = decisions[factor]
    lines.extend([
        f'### {factor.title()}', '',
        f"- Factor-equivalence decision: `{current['factor_equivalence_confirmed']}`",
        f"- Frozen-route mediation: `{current['route_mediation_confirmed']}`",
        f"- Shared target-dose slope: {interval(factor, 'shared_slope')}",
        f"- Raw target-context coefficient: {interval(factor, 'target_beta')}",
        f"- Raw source-context coefficient: {interval(factor, 'source_beta')}",
        f"- Source-answer coefficient: {interval(factor, 'source_answer_beta')}",
        f"- Mean raw-source progress toward target composites: {interval(factor, 'raw_progress_mean')}",
        f"- Source-context nuisance/progress ratio: {interval(factor, 'source_nuisance_to_progress_ratio')}",
        f"- Source-answer nuisance/progress ratio: {interval(factor, 'answer_nuisance_to_progress_ratio')}",
        f"- Shared minus residual slope: {interval(factor, 'shared_vs_residual_slope')}",
        f"- Shared minus random slope: {interval(factor, 'shared_vs_random_slope')}",
        f"- Shared minus residual counterfactual progress: {interval(factor, 'shared_vs_residual_progress')}",
        f"- Raw target-over-source-copy advantage: {interval(factor, 'raw_target_over_source_copy_advantage')}",
        f"- Route progress attenuation: {interval(factor, 'route_progress_attenuation')}",
        f"- Route progress specificity over random: {interval(factor, 'route_progress_specificity')}",
        '',
    ])
lines.extend([
    '## Interpretation boundary', '',
    (
        'A positive result supports a context-general, causally reusable '
        'factor signal whose downstream effect is consistent with combining '
        'that factor with information retained in the target scene. It does '
        'not show that the hidden state contains only that factor, that '
        'individual neurons are independent, or that no nuisance information '
        'can be decoded. If target dependence is present but source '
        'equivalence fails, the correct conclusion is functional '
        'recombination from a causally mixed state.'
    ), '',
    '## Audit', '',
    f"- Independent confirmation families: {audit['families']}",
    f"- Unique confirmation scenes: {audit['unique_confirmation_scenes']}",
    f"- Selection policy: `{audit['selection_policy']}`",
    f"- Physics violations: {audit['physics_violations']}",
    f"- Maximum self-replay error: {audit['maximum_self_replay_centered_error']:.6f}",
    f"- Maximum upstream norm error: {audit['maximum_upstream_equal_norm_relative_error']:.6f}",
    f"- Maximum route norm error: {audit['maximum_route_equal_norm_relative_error']:.6f}",
])
report_path = ANALYSIS_DIR / 'FACTOR_EQUIVALENCE_PRODUCT_INTERACTION_REPORT.md'
report_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')

paper_rows = []
for factor in FACTORS:
    row = {'factor': factor}
    for metric in (
        'shared_slope', 'target_beta', 'source_beta',
        'source_answer_beta', 'raw_progress_mean',
        'source_nuisance_to_progress_ratio',
        'answer_nuisance_to_progress_ratio', 'shared_vs_residual_slope',
        'shared_vs_random_slope', 'shared_vs_residual_progress',
        'raw_target_over_source_copy_advantage',
        'route_progress_attenuation', 'route_progress_specificity',
    ):
        current = test_frame[
            (test_frame['factor'] == factor)
            & (test_frame['metric'] == metric)
        ].iloc[0]
        row[f'{metric}_estimate'] = float(current['estimate'])
        row[f'{metric}_ci_low'] = float(current['ci_low'])
        row[f'{metric}_ci_high'] = float(current['ci_high'])
    row['factor_equivalence_confirmed'] = bool(
        decisions[factor]['factor_equivalence_confirmed']
    )
    row['route_mediation_confirmed'] = bool(
        decisions[factor]['route_mediation_confirmed']
    )
    paper_rows.append(row)
paper_table = pd.DataFrame(paper_rows)
paper_table.to_csv(ANALYSIS_DIR / 'paper_table.csv', index=False)

completion = {
    'schema_version': 'factor-equivalence-run-complete-1.0',
    'primary_decision': PRIMARY_DECISION,
    'model_id': MODEL_ID,
    'revision': MODEL_REVISION,
    'families': EXPECTED_FAMILIES,
    'records': int(len(record_frame)),
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'candidate_manifest_sha256': sha256_file(CANDIDATE_MANIFEST_PATH),
    'confirmation_manifest_sha256': sha256_file(CONFIRMATION_MANIFEST_PATH),
    'capture_sha256': CAPTURE_SHA256,
    'report': str(report_path),
    'figure': str(figure_path),
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
}
write_json_atomic(SHARED_ROOT / 'RUN_COMPLETE.json', completion)
display(paper_table.round(6))
print('Primary decision:', PRIMARY_DECISION)
print('Report:', report_path)
print('Run complete:', SHARED_ROOT / 'RUN_COMPLETE.json')
